# 2. Analytics Rules and Detection

**Analytics rules** are the detection engine of a SIEM: scheduled queries that run against your log data and fire *alerts* when patterns match.

### What you'll learn
- The six types of Sentinel analytics rules
- The bad → best progression for writing a rule (threshold, window, aggregation, entity mapping)
- How to tune a noisy rule (reduce false positives)
- MITRE ATT&CK coverage as a SOC maturity metric

## Types of Sentinel analytics rules

| Type | How it works | Latency | Use case |
|------|-------------|---------|----------|
| **Scheduled** | KQL runs on a schedule (e.g., every 5 min) | Minutes | Most detections |
| **NRT (Near Real-Time)** | KQL runs ~every minute | ~1 minute | Time-critical threats |
| **Microsoft Security** | Imports alerts from other Defender products | Seconds | XDR correlation |
| **Threat Intelligence** | Matches IOCs against log data | Minutes | Known-bad IPs, domains |
| **Anomaly** | ML-based baseline deviation | Varies | Unusual behavior |
| **Fusion** | Multi-stage attack correlation (ML) | Minutes | Advanced attacks |

## 0. Setup — pick the lab kernel

This lab has its own `uv`-managed virtual environment. Before running any code cell:

1. From `security-certs/sc-200/01-build-a-siem/` run once in a terminal:
   ```bash
   uv sync
   docker compose up -d
   ```
2. In VS Code, click the kernel picker (top-right of this notebook) and choose **`.venv (Python 3.xx)`** from this folder.
3. If the kernel does not appear, reload the window: `Cmd+Shift+P` → `Reload Window`.

The log-generator container has already seeded the SIEM with normal traffic **and** four attack patterns (brute force, lateral movement, exfiltration, phishing). Every cell below talks to `http://localhost:8000`.

In [ ]:
import httpx, json

SIEM = 'http://localhost:8000'

print('=== Current Analytics Rules ===')
rules = httpx.get(f'{SIEM}/rules').json()
for r in rules:
    sev = {'Critical':'🟣','High':'🔴','Medium':'🟡','Low':'🟢'}.get(r['severity'], '⬜')
    print(f'  {sev} [{r["id"]}] {r["name"]}')
    print(f'     Table: {r["query_table"]}  |  Tactic: {r["tactic"] or "-"}  |  Window: {r["window_minutes"]}min  |  Threshold: >={r["threshold"]}')
    if r['query_filter']: print(f'     Filter: {r["query_filter"]}')
    if r['aggregate_by']: print(f'     Group by: {r["aggregate_by"]}')
    print()

## 2.1 Bad rule → Best rule progression

New SOCs start with **naive** rules: alert on any failed sign-in, no threshold, no
grouping. Let's build the same detection three ways and compare what the SOC actually
receives.

Watch for the thing that is easy to miss: the naive rule's problem here is **not**
volume. Our engine emits one alert per rule run, so "bad" produces exactly as many
alerts as "best" does. Its problem is that the alert carries **no entity** — no user,
no host, no IP — so nothing downstream can use it. Incident correlation groups by
entity. Playbooks act on entities. An alert with no entity is a notification, not a
detection.

Volume is a *separate* failure mode, and it is controlled by the threshold. We measure
that immediately after, with numbers.

> Re-running these cells is safe — our mini-SIEM upserts rules by name. Note that each
> `/rules/evaluate` call re-fires **every** enabled rule, so alerts accumulate across
> runs. Real Sentinel behaves the same way; that is what *suppression* is for.

In [ ]:
# BAD: any single failure triggers an alert, with no grouping at all.
httpx.post(f'{SIEM}/rules', json={
    'name': 'Demo - failed sign-in (bad)',
    'severity': 'Low',
    'tactic': 'CredentialAccess',
    'query_table': 'SigninLogs',
    'query_filter': {'ResultType': 'Failure'},
    'threshold': 1,
    'window_minutes': 1440,
    'description': 'One alert covering every failed sign-in in the tenant. No entity, so no incident correlation and no playbook can act on it.',
})

# BETTER: add a threshold so isolated typos dont alert
httpx.post(f'{SIEM}/rules', json={
    'name': 'Demo - failed sign-in (better)',
    'severity': 'Medium',
    'tactic': 'CredentialAccess',
    'query_table': 'SigninLogs',
    'query_filter': {'ResultType': 'Failure'},
    'threshold': 10,
    'window_minutes': 60,
    'description': '10+ failures org-wide in 1h. Suppresses small noise but still one aggregate count - it cannot tell you WHO.',
})

# BEST: aggregate per user, high severity, mapped to ATT&CK
httpx.post(f'{SIEM}/rules', json={
    'name': 'Demo - failed sign-in (best)',
    'severity': 'High',
    'tactic': 'CredentialAccess',
    'query_table': 'SigninLogs',
    'query_filter': {'ResultType': 'Failure'},
    'aggregate_by': 'UserPrincipalName',
    'threshold': 5,
    'window_minutes': 60,
    'description': '5+ failures per user in 1h. Entity = user -> feeds incident correlation and playbooks.',
})
print('Created 3 demo rules: bad / better / best.')


In [ ]:
# Evaluate and compare what each rule handed the SOC
r = httpx.post(f'{SIEM}/rules/evaluate').json()
demo = {a['rule']: a for a in r['alerts_created'] if a['rule'].startswith('Demo - ')}
demo_alerts = [a for a in r['alerts_created'] if a['rule'].startswith('Demo - ')]

print(f'Demo alerts fired: {len(demo_alerts)}\n')
print(f'{"rule":<34} {"alerts":>6}  {"events":>6}  entity')
print('-' * 78)
for name in ('Demo - failed sign-in (bad)', 'Demo - failed sign-in (better)', 'Demo - failed sign-in (best)'):
    hits = [a for a in demo_alerts if a['rule'] == name]
    ents = [h['group'] for h in hits if 'group' in h] or ['<none>']
    print(f'{name:<34} {len(hits):>6}  {sum(h["count"] for h in hits):>6}  {", ".join(ents)}')

bad = [a for a in demo_alerts if a['rule'].endswith('(bad)')]
better = [a for a in demo_alerts if a['rule'].endswith('(better)')]
best = [a for a in demo_alerts if a['rule'].endswith('(best)')]

# The lesson is entity mapping, so assert exactly that: bad/better name nobody,
# best names the affected account.
assert bad and better and best, f'all three demo rules should fire on this data, got {sorted(demo)}'
assert all('group' not in a for a in bad + better), 'bad/better must produce entity-less alerts'
assert all('group' in a for a in best), 'best must attach an entity to every alert'
assert {a['group'] for a in best} == {'alice@contoso.com'}, \
    f"best rule should name only the brute-forced account, got {sorted(a['group'] for a in best)}"

print('\nKey takeaway:')
print(f'  bad    -> 1 alert covering {bad[0]["count"]} failures across the whole tenant, entity: none')
print(f'  better -> 1 alert covering {better[0]["count"]} failures, entity: still none - you know a number, not a victim')
print(f'  best   -> 1 alert per affected user ({", ".join(a["group"] for a in best)}) - the SOC can act on it')
print('\nSame data, same alert count. Only the BEST rule produces something an')
print('incident, a playbook, or an analyst can pivot on.')


### Now measure it: precision vs the threshold

The three rules above differ in *shape*. The threshold is a separate dial, and it is the
one that decides how much of the SOC's day is wasted. A rule that fires on the seeded
attack is easy; a rule that fires on the attack **and not on the four people who
mistyped their password** is the actual job.

To measure that you need ground truth. In production you get it from analyst
dispositions — the `TruePositive` / `BenignPositive` / `FalsePositive` classifications
you will write in notebook 3. Here we derive it from an independent signal that is in
the data but *not* in the rule: the brute-force attempts came from an external address,
the typos came from corporate `10.x` ranges.

In [ ]:
# Ground truth, derived from a signal the rule under test does NOT look at:
# a genuinely attacked account has failures from a non-corporate source address.
sig = httpx.post(f'{SIEM}/query', json={
    'table_name': 'SigninLogs',
    'filter': {'ResultType': 'Failure'},
    'limit': 1000,
}).json()['results']

failures_by_user, attacked = {}, set()
for s in sig:
    failures_by_user[s['UserPrincipalName']] = failures_by_user.get(s['UserPrincipalName'], 0) + 1
    if not s['IPAddress'].startswith('10.'):
        attacked.add(s['UserPrincipalName'])

print(f'Users with failed sign-ins: {len(failures_by_user)}   truly attacked: {sorted(attacked)}\n')
print(f'{"threshold":>9} {"alerts":>7} {"TP":>3} {"FP":>3} {"precision":>10} {"recall":>7}   who fires')
print('-' * 88)

rows = []
for threshold in range(1, 8):
    firing = {u for u, c in failures_by_user.items() if c >= threshold}
    tp = len(firing & attacked)
    fp = len(firing - attacked)
    precision = tp / len(firing) if firing else 0.0
    recall = tp / len(attacked) if attacked else 0.0
    rows.append((threshold, len(firing), tp, fp, precision, recall))
    who = ', '.join(sorted(firing)) or '-'
    print(f'{threshold:>9} {len(firing):>7} {tp:>3} {fp:>3} {precision:>9.0%} {recall:>7.0%}   {who}')

# The whole point: a lower threshold does NOT find more attacks here, it only finds
# more innocent people. If that were not true, tuning would be a trade-off instead of
# a free win - and the lab would be teaching you a false comfort.
naive = next(r for r in rows if r[0] == 1)
tuned = next(r for r in rows if r[0] == 5)
assert naive[3] > 0, 'threshold=1 must generate false positives, or there is no benign population in the data'
assert tuned[3] == 0, f'threshold=5 should be clean, got {tuned[3]} false positive(s)'
assert tuned[4] > naive[4], f'tuning must improve precision ({naive[4]:.0%} -> {tuned[4]:.0%})'
assert tuned[5] == naive[5] == 1.0, 'recall must be unchanged - otherwise we traded away detections'

print(f'\nthreshold 1 -> {naive[4]:.0%} precision ({naive[3]} innocent accounts alerted on)')
print(f'threshold 5 -> {tuned[4]:.0%} precision, recall unchanged at {tuned[5]:.0%}')
print('\nThat is the shape of a good tuning decision: false positives collapse, true')
print('positives survive. If recall had dropped too, you would be hiding attacks, and')
print('you would need a second signal (impossible travel, new device, risk level)')
print('rather than a bigger number.')


## 2.2 Tuning a noisy rule — real-world false-positive reduction

A detection that constantly alerts on normal behaviour is worse than no detection at all. Watch what happens when we create an over-broad rule, then tune it:

In [ ]:
# Noisy rule: any new process on the DB server. Looks strict, but normal admins run things too.
httpx.post(f'{SIEM}/rules', json={
    'name': 'DB-server new process (over-broad)',
    'severity': 'High',
    'tactic': 'Execution',
    'query_table': 'DeviceEvents',
    'query_filter': {'DeviceName': 'vm-db-01', 'ActionType': 'ProcessCreated'},
    'aggregate_by': 'FileName',
    'threshold': 1,
    'window_minutes': 120,
    'description': 'T1059 - any process launch on the DB server. Deliberately over-broad.',
})
r = httpx.post(f'{SIEM}/rules/evaluate').json()
db_alerts = [a for a in r['alerts_created'] if a['rule'] == 'DB-server new process (over-broad)']

# Ground truth for this rule: attacker tooling vs everyday developer/admin binaries.
ATTACK_TOOLS = {'mimikatz.exe', 'psexec.exe', 'certutil.exe', 'nc.exe', 'cmd.exe'}

print(f'Over-broad rule fired {len(db_alerts)} alerts:')
tp = fp = 0
for a in sorted(db_alerts, key=lambda x: x['group']):
    is_tp = a['group'] in ATTACK_TOOLS
    tp, fp = tp + is_tp, fp + (not is_tp)
    print(f'  [{"TP" if is_tp else "FP"}] {a["group"]:<15} ({a["count"]} events)')

overbroad_precision = tp / len(db_alerts) if db_alerts else 0.0
print(f'\nPrecision: {tp} TP / {len(db_alerts)} alerts = {overbroad_precision:.0%}')
print(f'{fp} of those alerts are developer noise on that host, not an attack.')

# If this rule stopped being noisy the tuning demo below would be demonstrating nothing.
assert fp >= 1, 'the over-broad rule must produce false positives for this section to teach anything'
assert 'mimikatz.exe' in {a['group'] for a in db_alerts}, 'the over-broad rule must still catch the real threat'
assert overbroad_precision < 1.0, f'expected an imprecise rule, got {overbroad_precision:.0%}'


In [ ]:
# Tuned rule: only alert on known attacker tooling. Watch the precision, not the vibe.
# NOTE the tactic. Mimikatz is T1003 OS Credential Dumping, so this is a CREDENTIAL
# ACCESS detection even though the observable is a process launch. Sentinel's built-in
# Mimikatz rules are tagged the same way; tagging it "Execution" because the log row
# says ProcessCreated is a classic exam trap.
httpx.post(f'{SIEM}/rules', json={
    'name': 'DB-server new process (tuned)',
    'severity': 'High',
    'tactic': 'CredentialAccess',
    'query_table': 'DeviceEvents',
    'query_filter': {'DeviceName': 'vm-db-01', 'FileName': 'mimikatz.exe'},
    'threshold': 1,
    'window_minutes': 120,
    'description': 'T1003 OS Credential Dumping. Only known-bad binaries. Combine with an allowlist watchlist in real Sentinel.',
})
r = httpx.post(f'{SIEM}/rules/evaluate').json()
tuned = [a for a in r['alerts_created'] if a['rule'] == 'DB-server new process (tuned)']
tuned_precision = 1.0 if tuned else 0.0   # every alert this rule can raise is on mimikatz

print(f'Tuned rule fired {len(tuned)} alert(s) - just the real threat.')
print(f'Precision: {overbroad_precision:.0%} -> {tuned_precision:.0%}   '
      f'(alerts per evaluation: {len(db_alerts)} -> {len(tuned)})')

assert len(tuned) >= 1, 'the tuned rule must still detect the mimikatz execution'
assert tuned_precision > overbroad_precision, 'tuning must improve precision, not just reduce volume'
assert len(tuned) < len(db_alerts), 'tuning must also reduce the alert volume'

# Clean up the over-broad rule so it stops producing noise
over_broad = next((x for x in httpx.get(f'{SIEM}/rules').json() if x['name']=='DB-server new process (over-broad)'), None)
if over_broad:
    httpx.delete(f'{SIEM}/rules/{over_broad["id"]}')
    print(f'Deleted over-broad rule {over_broad["id"]}.')

print('\n⚠️  Notice what tuning just cost us: the over-broad rule was our only')
print('   Execution-tactic detection, and its replacement is a Credential Access one.')
print('   The coverage map below will show the hole. That is not a bug in the lab -')
print('   it is why you re-check coverage after every tuning change.')


## 2.3 MITRE ATT&CK coverage

Every rule should map to a MITRE ATT&CK tactic so you can see where in the adversary lifecycle you have visibility.

```
Reconnaissance -> Resource Development -> Initial Access -> Execution -> Persistence ->
Privilege Escalation -> Defense Evasion -> Credential Access -> Discovery ->
Lateral Movement -> Collection -> Command & Control -> Exfiltration -> Impact
```

In [ ]:
# Sentinel's tactic enum - a rule tagged with anything outside this list is not
# actually mapped to ATT&CK, it just looks like it is in the portal.
MITRE_TACTICS = [
    'Reconnaissance','ResourceDevelopment','InitialAccess','Execution',
    'Persistence','PrivilegeEscalation','DefenseEvasion','CredentialAccess',
    'Discovery','LateralMovement','Collection','CommandAndControl',
    'Exfiltration','Impact',
]
rules = httpx.get(f'{SIEM}/rules').json()
tagged = {r['tactic'] for r in rules if r['tactic']}

# Only tactics that are BOTH used by a rule and real ATT&CK tactics count as covered.
# Counting `tagged` directly lets a typo ("LateralMovment") inflate your coverage.
covered = tagged & set(MITRE_TACTICS)
invalid = tagged - set(MITRE_TACTICS)

print('=== MITRE ATT&CK Coverage ===\n')
for tactic in MITRE_TACTICS:
    matching = [r['name'] for r in rules if r['tactic'] == tactic]
    if matching:
        print(f'  ✅ {tactic}')
        for name in matching:
            print(f'       - {name}')
    else:
        print(f'  ⬜ {tactic} - no detection')

pct = len(covered)/len(MITRE_TACTICS)*100
print(f'\nCoverage: {len(covered)}/{len(MITRE_TACTICS)} tactics ({pct:.0f}%)')

assert not invalid, f'rules tagged with tactics that are not ATT&CK tactics: {sorted(invalid)}'
assert {'InitialAccess', 'CredentialAccess', 'LateralMovement', 'Exfiltration'} <= covered, \
    f'the seeded kill chain should be covered end to end, missing: ' \
    f'{sorted({"InitialAccess","CredentialAccess","LateralMovement","Exfiltration"} - covered)}'

gaps = [t for t in MITRE_TACTICS if t not in covered]
print(f'\n{len(gaps)} tactics with no detection at all: {", ".join(gaps)}')
print('\n💡 Read this map as a to-do list, not a scorecard. Two honest observations:')
print('   1. Execution is uncovered because tuning replaced our only Execution rule')
print('      with a Credential Access one. Coverage regressions hide inside "we fixed')
print('      the noisy rule" changes.')
print('   2. 100% coverage is not the goal and is not achievable - Reconnaissance and')
print('      Resource Development happen on infrastructure you cannot see. Aim for')
print('      depth on the tactics your telemetry can actually observe.')


## 2.4 View generated alerts

Each alert carries:
- **Severity** — business impact
- **Tactic** — adversary-lifecycle stage
- **Entities** — *who* or *what* (user, host, IP)
- **Evidence** — sample events that triggered the detection

In [ ]:
alerts = httpx.get(f'{SIEM}/alerts').json()
print(f'=== All Alerts ({len(alerts)} total) ===\n')
for a in alerts[:10]:
    sev = {'Critical':'🟣','High':'🔴','Medium':'🟡','Low':'🟢'}.get(a['severity'], '⬜')
    entities = json.loads(a['entities']) if isinstance(a['entities'], str) else a['entities']
    print(f'{sev} [{a["status"]}] {a["title"]}')
    if entities: print(f'   Entities: {entities}')
    print(f'   Tactic: {a["tactic"]}  |  Created: {a["created_at"]}\n')

### SC-200 exam: analytics rule fields

| Field | What it controls |
|-------|-----------------|
| **Query** | KQL that identifies the threat |
| **Query frequency** | How often the rule runs (e.g., every 5 min) |
| **Query lookback** | How far back to search (e.g., last 1 hour) |
| **Trigger threshold** | Minimum result count to fire (e.g., > 0) |
| **Entity mapping** | Which fields are accounts, IPs, hosts |
| **MITRE tactics** | ATT&CK classification |
| **Alert grouping** | Group related alerts (by entity, time, ...) |
| **Event grouping** | How many events attach per alert |
| **Suppression** | Silence the rule after it fires (e.g., 1h) |

**Next**: [Notebook 3 — Incidents and Automation](03_incidents_and_automation.ipynb)